This simulation is based on the paper -- "Quantum Neuronal Sensing of Quantum Many-Body States on a 61-Qubit Programmable Superconducting Processor"
Please refer to https://github.com/jd-opensource/TeD-Q/blob/main/examples/Many_body_Localization_1D.ipynb

In [ ]:
# =========================
# Many-Body Localization (1D) — MindQuantum version
# (drop-in replacement for the "circuit part" + measurement)
# =========================

import math
import numpy as np

# MindQuantum imports
from mindquantum import Circuit
from mindquantum.simulator import Simulator

# -------------------------
# Init parameters (same semantics as your original)
# -------------------------
n_size = 3
n_epoch = 100
n_train = 200
n_test = 50

lr = 0.01
gamma_lr_scheduler = 0.2

n_qubits = n_size
t_d = 200e-9  # ns
h_bar = 1.0
g = 2.185e6   # MHz
h_erg = 1e6   # MHz
h_loc = 40e6  # MHz

# Derived constant used inside H0 (your original used torch.Tensor([g*h_bar*t_d]))
ANGLE_CONST = float(g * h_bar * t_d)

# -------------------------
# Helper (kept for parity; not strictly needed in 1D)
# -------------------------
def getIndex(i, j):
    return n_size * i + j

# -------------------------
# Gate blocks: H0 and Hd (MindQuantum)
# -------------------------
def H0(c: Circuit, idx: int, jdx: int):
    """Coupling block matching your original H0 structure."""
    # Block 1
    c.h(idx)
    c.h(jdx)
    c.x(idx, jdx)
    c.rz(ANGLE_CONST, jdx)
    c.x(idx, jdx)
    c.h(idx)
    c.h(jdx)
    # Block 2
    c.s(idx)
    c.s(jdx)
    c.h(idx)
    c.h(jdx)
    c.x(idx, jdx)
    c.rz(ANGLE_CONST, jdx)
    c.x(idx, jdx)
    c.h(idx)
    c.h(jdx)
    # PhaseShift(-pi/2) ≡ RZ(-pi/2) up to global phase
    c.rz(-math.pi / 2.0, idx)
    c.rz(-math.pi / 2.0, jdx)

def Hd(c: Circuit, idx: int, jdx: int, di):
    """Disorder on jdx then fixed H0 coupling for (idx, jdx)."""
    c.rz(float(di), jdx)
    H0(c, idx, jdx)

def partial_connected(c: Circuit):
    """Sparse extra connectivity to increase complexity (kept as in your code)."""
    for i in range(0, n_qubits - 7, 5):
        c.x(i, i + 7)

# -------------------------
# Build full circuit (MindQuantum)
# -------------------------
def build_mq_circuit(d, params):
    """
    d: shape (N,) disorder angles, where N = (n_size - 1) * 2
    params: shape (n_qubits + 1, 2), per-qubit [theta, phi] + final-qubit head
    returns: (Circuit, [measured_qubits])
    """
    d = np.asarray(d, dtype=float).reshape(-1)
    params = np.asarray(params, dtype=float)

    c = Circuit()

    # 1) Néel state: X on even sites
    for i in range(n_qubits):
        if i % 2 == 0:
            c.x(i)

    # 2) Hd blocks over 1-D nearest neighbors (both directions, matching your loops)
    cnt = 0
    for i in range(n_qubits):
        if i + 1 < n_qubits:
            Hd(c, i + 1, i, d[cnt]); cnt += 1
        if i - 1 >= 0:
            Hd(c, i - 1, i, d[cnt]); cnt += 1

    # 3) Variational single-qubit layer: RZ(θ) RX(φ) RZ(-θ)
    for i in range(n_qubits):
        theta = float(params[i, 0])
        phi   = float(params[i, 1])
        c.rz(theta, i)
        c.rx(phi, i)
        c.rz(-theta, i)

    # 4) Extra sparse connectivity
    partial_connected(c)

    # 5) H0 nearest-neighbor couplings again (your original had range over n_size)
    for i in range(n_size):
        if i + 1 < n_qubits:
            H0(c, i + 1, i)
        if i - 1 >= 0:
            H0(c, i - 1, i)

    # 6) Last-qubit head: RZ(θL) RX(φL) RZ(-θL)
    last = n_qubits - 1
    thetaL = float(params[n_qubits, 0])
    phiL   = float(params[n_qubits, 1])
    c.rz(thetaL, last)
    c.rx(phiL, last)
    c.rz(-thetaL, last)

    measured_qubits = [last]
    return c, measured_qubits

# -------------------------
# Measurement utilities
# -------------------------
def probs_last_qubit_from_statevector(state, last_qubit: int):
    """
    Exact P(q_last = 0/1) from statevector.
    MindQuantum uses little-endian basis: bit of qubit k in basis index is ((i >> k) & 1).
    """
    state = np.asarray(state)
    n = int(round(np.log2(state.size)))
    p0 = 0.0
    p1 = 0.0
    for i, amp in enumerate(state):
        bit = (i >> last_qubit) & 1
        prob = (amp.real * amp.real + amp.imag * amp.imag)
        if bit == 0:
            p0 += prob
        else:
            p1 += prob
    # numerical hygiene
    s = p0 + p1
    if s > 0:
        p0 /= s
        p1 /= s
    return np.array([p0, p1], dtype=float)

def measure_last_qubit_probs(d, params, shots=None, backend='mqvector'):
    """
    Build circuit, run simulator, and return [P0, P1] on the last qubit.
      - shots=None  -> exact (statevector)
      - shots=int   -> sampling estimate with given shots
    """
    circ, measured = build_mq_circuit(d, params)
    last = measured[0]

    sim = Simulator(backend, n_qubits)

    if shots is None:
        # exact
        sim.apply_circuit(circ)
        state = sim.get_qs()  # complex np.ndarray of length 2**n
        return probs_last_qubit_from_statevector(state, last)
    else:
        # shot-based estimate
        samp = sim.sampling(circ, shots=shots)  # dict: {bitstring: count}
        c0, c1 = 0, 0
        for bitstring, cnt in samp.items():
            # MindQuantum's bitstring is little-endian with qubit-0 as the rightmost char.
            # last qubit (index n_qubits-1) corresponds to the LEFT-most char.
            last_bit_char = bitstring[0]
            if last_bit_char == '0':
                c0 += cnt
            else:
                c1 += cnt
        total = max(1, c0 + c1)
        return np.array([c0 / total, c1 / total], dtype=float)

# -------------------------
# Dataset helpers (same distributions as your original)
# -------------------------
def make_train_dataset():
    """
    Returns:
      d (n_train, N) disorder angles
      y_target (n_train,) labels with 1 for ergodic, 0 for localized
    """
    N = (n_size - 1) * 2
    # ergodic: uniform in [-1, 1] scaled by h_erg * h_bar * t_d * pi
    d_erg = (np.random.rand(n_train // 2, N) * 2.0 - 1.0) * (h_erg * h_bar * t_d * math.pi)

    # localized: original used ((rand*39/40 + 1/40) * sign) * h_loc * ... * pi
    core = (np.random.rand(n_train // 2, N) * 39.0 / 40.0 + 1.0 / 40.0)
    signs = np.random.choice([-1.0, 1.0], size=(n_train // 2, N))
    d_local = (core * signs) * (h_loc * h_bar * t_d * math.pi)

    d = np.concatenate([d_erg, d_local], axis=0)
    y_target = np.array([1] * (n_train // 2) + [0] * (n_train // 2), dtype=float)
    return d, y_target

def make_test_dataset():
    N = (n_size - 1) * 2
    d_erg = (np.random.rand(n_test // 2, N) * 2.0 - 1.0) * (h_erg * h_bar * t_d * math.pi)
    # for test, your snippet showed a plain uniform for localized (keep parity with your log)
    d_local = (np.random.rand(n_test // 2, N) * 2.0 - 1.0) * (h_loc * h_bar * t_d * math.pi)
    d = np.concatenate([d_erg, d_local], axis=0)
    y_target = np.array([1] * (n_test // 2) + [0] * (n_test // 2), dtype=float)
    return d, y_target

# -------------------------
# Example: build one batch of y (exact or shots)
# -------------------------
if __name__ == "__main__":
    # shapes consistent with your original
    N = (n_size - 1) * 2
    # trainable params (theta, phi) per qubit + last-qubit head
    params = np.random.rand(n_qubits + 1, 2).astype(float)

    # training dataset
    d_train, y_train = make_train_dataset()

    # compute probabilities y for the last qubit (exact)
    # y_exact will be shape (n_train, 2), each row is [P0, P1]
    y_exact = np.stack(
        [measure_last_qubit_probs(di, params, shots=None, backend='mqvector') for di in d_train],
        axis=0
    )

    print("y_exact shape:", y_exact.shape, "(rows = samples, cols = [P0, P1])")
    print("first 3 rows:\n", y_exact[:3])

    # If you want shot-based estimates instead (e.g., 20000 shots):
    # y_shot = np.stack(
    #     [measure_last_qubit_probs(di, params, shots=20000, backend='mqvector') for di in d_train],
    #     axis=0
    # )
    # print("y_shot shape:", y_shot.shape)


In [ ]:
# shapes consistent with your original
N = (n_size - 1) * 2
# trainable params (theta, phi) per qubit + last-qubit head
params = np.random.rand(n_qubits + 1, 2).astype(float)

# training dataset
d_train, y_train = make_train_dataset()

Using State Vector Simulator

In [ ]:
circ, measured = build_mq_circuit(d, params)
last = measured[0]


Time for calculation using State Vector simulator

In [ ]:
%%timeit
sim = Simulator(backend, n_qubits)
sim.apply_circuit(circ)
state = sim.get_qs()  # complex np.ndarray of length 2**n
probs_last_qubit_from_statevector(state, last)

In [ ]:
sim = Simulator(backend, n_qubits)
sim.apply_circuit(circ)
state = sim.get_qs()  # complex np.ndarray of length 2**n
a = probs_last_qubit_from_statevector(state, last)
a

Using Tensor Network simulator

In [ ]:
from mindquantum import TN_Circuit
import cotengra as ctg
from quimb.tensor import Tensor

In [ ]:
tn_circ = TN_Circuit(circ)

In [ ]:
last_qubit = n_qubits - 1

In [ ]:
ket = tn_circ.psi
bra = ket.H
bra.reindex_({f'k{i}': f'b{i}' for i in range(n)})
tn = bra & ket

In [ ]:
import numpy as np
I2 = np.eye(2).reshape(2, 2)
for i in range(n):
    if i == last_qubit:
        continue
    tn.add_tensor(Tensor(data=I2, inds=(f'k{i}', f'b{i}')))

In [ ]:
output_inds = tn.outer_inds()

In [ ]:
# inplace full simplify and cast to single precision
tn.full_simplify_(output_inds=output_inds)
tn.astype_("complex64")

In [ ]:
tn.graph(
    initial_layout="kamada_kawai",
    iterations=10,
    color=tn_circ.psi.site_tags,
    legend=False,
    figsize=(3, 3),
)

In [ ]:
opt = ctg.HyperOptimizer(
    methods=["kahypar", "greedy", "walktrap"],
    max_repeats=128,
    progbar=True,
    minimize="flops",
    score_compression=0.5,  # deliberately make the optimizer try many methods
)

In [ ]:
tree = opt.get_tree()
tree.plot_ring(node_scale=1 / 3, edge_scale=2 / 3)

In [ ]:
tree.plot_tent()

In [ ]:
%%timeit
tn.contract(all, optimize=opt.path, output_inds=output_inds)

In [ ]:
a = tn.contract(all, optimize=opt.path, output_inds=output_inds)
a.data